In [36]:
# import cell
from pathlib import Path
from collections.abc import Sequence
import csv
import math
import re
import cv2
import glob
import os

import matplotlib.pyplot as plt
from matplotlib.colors import rgb_to_hsv
import numpy as np
from PIL import Image

In [48]:
def shades_of_gray(img, p=6):
    """
    Shades of Gray
    """
    img_float = img.astype(np.float32) / 255.0
    b, g, r = cv2.split(img_float)

    #  L = (mean(I^p))^(1/p)
    l_b = np.power(np.mean(np.power(b, p)), 1.0 / p)
    l_g = np.power(np.mean(np.power(g, p)), 1.0 / p)
    l_r = np.power(np.mean(np.power(r, p)), 1.0 / p)

    if l_b == 0 or l_g == 0 or l_r == 0:
        return img.copy()

    l_max = np.max([l_b, l_g, l_r])

    gain_b = l_max / l_b
    gain_g = l_max / l_g
    gain_r = l_max / l_r

    b_corrected = b * gain_b * 255.0
    g_corrected = g * gain_g * 255.0
    r_corrected = r * gain_r * 255.0

    img_corrected = cv2.merge([b_corrected, g_corrected, r_corrected])
    img_corrected = np.clip(img_corrected, 0, 255).astype(np.uint8)
    
    return img_corrected
    
def dull_razor_hair_removal(img):
    """
    Step 2: Dull-Razor 算法 (精准识别毛发并擦除) //引用自博客园
    """
    # 1. 转为灰度图 (修正：OpenCV 读取的图是 BGR，所以用 BGR2GRAY)
    grayScale = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    # 2. 形态学黑帽变换 (保留你的设计：使用 10x10 的十字形内核 MORPH_CROSS)
    kernel = cv2.getStructuringElement(cv2.MORPH_CROSS, (10, 10))
    blackhat = cv2.morphologyEx(grayScale, cv2.MORPH_BLACKHAT, kernel)
    
    # 3. 阈值分割得到二值化掩膜
    _, thresh2 = cv2.threshold(blackhat, 10, 255, cv2.THRESH_BINARY)
    
    # 4. 图像修复
    dst = cv2.inpaint(img, thresh2, 1, cv2.INPAINT_TELEA)
    
    return dst, thresh2
    
def pipeline_preview(input_dir, output_dir=r"B:\Project\Data_Proj2\processed"):

    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    extensions = ('*.jpg', '*.jpeg', '*.png', '*.bmp', '*.JPG', '*.JPEG', '*.PNG')
    image_paths = []
    for ext in extensions:
        image_paths.extend(glob.glob(os.path.join(input_dir, ext)))
    
    total_images = len(image_paths)
    if total_images == 0:
        print(f"提示: 在文件夹 '{input_dir}' 中没有找到任何图片。")
        return

    print(f"成功检索到 {total_images} 张图片")
    
    # 3. 循环遍历处理每一张图片
    for index, image_path in enumerate(image_paths, start=1):
        base_name = os.path.basename(image_path)
        print(f"[{index}/{total_images}] 正在处理: {base_name} ...")
        
        # 读取图像
        img = cv2.imread(image_path)
        if img is None:
            print(f"   ⚠️ 警告: 无法读取图片 {base_name}，跳过该图。")
            continue
            
        try:
            # 串联流水线
            img_cc = shades_of_gray(img, p=6)  # Step 1: 颜色校正
            img_blur = cv2.medianBlur(img_cc, 3)
            img_final, hair_mask = dull_razor_hair_removal(img_blur)  # Step 3: 拔毛
            
            # 创建四宫格对比图进行效果追踪
            mask_3ch = cv2.cvtColor(hair_mask, cv2.COLOR_GRAY2BGR)
            top_row = np.hstack((img, img_cc))
            bottom_row = np.hstack((mask_3ch, img_final))
            quad_comparison = np.vstack((top_row, bottom_row))

            # 保存中间及最终结果
            cv2.imwrite(os.path.join(output_dir, f"{base_name}"), img_final)
            cv2.imwrite(os.path.join(output_dir, f"comparison_quad_{base_name}"), quad_comparison)
    
        except Exception as e:
            print(f"   ❌ 错误: 处理图片 {base_name} 时发生异常: {e}")
            continue

    print("="*50 + f"\n 批量处理全部完成！所有结果已保存至: {output_dir}")


In [49]:
# --- 独立运行入口 ---
if __name__ == "__main__":
    # 使用时请将此处的路径替换为你电脑上的皮肤病图片路径
    your_image_path = r"B:\Project\Data_Proj2\image" 
        
    # 运行 Pipeline
    pipeline_preview(your_image_path)

成功检索到 1200 张图片，开始批量处理...
[1/1200] 正在处理: 1.jpg ...
[2/1200] 正在处理: 10.jpg ...
[3/1200] 正在处理: 100.jpg ...
[4/1200] 正在处理: 100_aug1.jpg ...
[5/1200] 正在处理: 100_aug2.jpg ...
[6/1200] 正在处理: 101.jpg ...
[7/1200] 正在处理: 101_aug1.jpg ...
[8/1200] 正在处理: 101_aug2.jpg ...
[9/1200] 正在处理: 102.jpg ...
[10/1200] 正在处理: 102_aug1.jpg ...
[11/1200] 正在处理: 102_aug2.jpg ...
[12/1200] 正在处理: 103.jpg ...
[13/1200] 正在处理: 103_aug1.jpg ...
[14/1200] 正在处理: 103_aug2.jpg ...
[15/1200] 正在处理: 104.jpg ...
[16/1200] 正在处理: 104_aug1.jpg ...
[17/1200] 正在处理: 104_aug2.jpg ...
[18/1200] 正在处理: 105.jpg ...
[19/1200] 正在处理: 105_aug1.jpg ...
[20/1200] 正在处理: 105_aug2.jpg ...
[21/1200] 正在处理: 106.jpg ...
[22/1200] 正在处理: 106_aug1.jpg ...
[23/1200] 正在处理: 106_aug2.jpg ...
[24/1200] 正在处理: 107.jpg ...
[25/1200] 正在处理: 107_aug1.jpg ...
[26/1200] 正在处理: 107_aug2.jpg ...
[27/1200] 正在处理: 108.jpg ...
[28/1200] 正在处理: 108_aug1.jpg ...
[29/1200] 正在处理: 108_aug2.jpg ...
[30/1200] 正在处理: 109.jpg ...
[31/1200] 正在处理: 109_aug1.jpg ...
[32/1200] 正在处理: 109_aug2

In [ ]:
# 优化去除毛发的步骤：先监测是否有毛发，如果没有毛发就不处理，有毛发再去除。
# 没有毛发：print("没有毛发")
# 有毛发：print("有毛发，正在去除...")

In [ ]:
# 在此基础上，读取 image 文件夹中的一个图片，进行毛发去除处理，并显示原图和处理后的图像。
